# MMSD: fine-tuned ResNet-50 V2 image features


In [ ]:
%pip install timm numpy Pillow tqdm


In [ ]:
import json
import os
import platform
import shutil
import time
from pathlib import Path

import numpy as np
import PIL
from PIL import Image
import torch
import torchvision
import timm
from tqdm.auto import tqdm

PART = 3
NUM_PARTS = 3
DATA_ROOT = Path('/kaggle/input/datasets/fruich/mmsd-full/data')
IMAGE_DIR = DATA_ROOT / 'raw/dataset_image'
OUTPUT_DIR = Path('/kaggle/working/features/resnetv2_50_a1h_in1k_finetuned')
CHECKPOINT = Path('/kaggle/input/your-checkpoint-dataset/finetuned_resnet50v2.pt')
assert 1 <= PART <= NUM_PARTS
assert CHECKPOINT.is_file(), f'Run train_resnet50v2_image_classifier.ipynb first: {CHECKPOINT}'
BATCH_SIZE = 32
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, 'Device:', DEVICE, 'GPUs:', torch.cuda.device_count())


In [ ]:
all_tweet_ids = sorted(path.stem for path in IMAGE_DIR.glob('*.jpg'))
tweet_ids = [str(i) for i in np.array_split(all_tweet_ids, NUM_PARTS)[PART - 1]]
print('Total:', len(all_tweet_ids), 'This part:', PART, len(tweet_ids))
print('Feature storage:', len(tweet_ids) * 196 * 2048 * 4 / 1e9, 'GB plus headers')


In [ ]:
MODEL_NAME = 'resnetv2_50.a1h_in1k'
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=1)
checkpoint_data = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
model.load_state_dict(checkpoint_data['model'])
model.requires_grad_(False).eval().to(DEVICE)
mean = torch.tensor(model.pretrained_cfg['mean'], dtype=torch.float32).view(1, 3, 1, 1)
std = torch.tensor(model.pretrained_cfg['std'], dtype=torch.float32).view(1, 3, 1, 1)

class SpatialEncoder(torch.nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, inputs):
        return self.backbone.forward_features(inputs)

encoder = SpatialEncoder(model).eval()
if torch.cuda.device_count() > 1:
    encoder = torch.nn.DataParallel(encoder)

@torch.inference_mode()
def extract_batch(ids):
    images = []
    for tweet_id in ids:
        with Image.open(IMAGE_DIR / f'{tweet_id}.jpg') as image:
            image = image.convert('RGB').resize((448, 448), Image.Resampling.BICUBIC)
            images.append(np.asarray(image, dtype=np.float32))
    inputs = torch.from_numpy(np.stack(images)).permute(0, 3, 1, 2) / 255.0
    inputs = ((inputs - mean) / std).to(DEVICE)
    spatial = encoder(inputs)
    assert tuple(spatial.shape) == (len(ids), 2048, 14, 14), spatial.shape
    features = spatial.flatten(2).transpose(1, 2).contiguous().cpu().numpy()
    assert features.dtype == np.float32 and np.isfinite(features).all()
    return features


In [ ]:
manifest = {
    'upstream_revision': 'f42b16510208624d91fa545ca9bb64c6335f971e',
    'encoder': MODEL_NAME, 'weights': 'fine-tuned checkpoint initialized from timm ImageNet-1k a1h',
    'checkpoint': str(CHECKPOINT), 'checkpoint_epoch': checkpoint_data['epoch'],
    'checkpoint_validation_f1': checkpoint_data['validation_f1'],
    'feature_layer': 'forward_features (after final normalization)', 'backbone_frozen_during_extraction': True,
    'resize': [448, 448], 'color': 'RGB', 'resampling': 'Pillow BICUBIC',
    'preprocessing': 'direct resize without cropping; RGB / 255, then channel normalization',
    'mean': list(model.pretrained_cfg['mean']), 'std': list(model.pretrained_cfg['std']),
    'output_shape': [196, 2048], 'dtype': 'float32',
    'flatten_order': 'row-major spatial grid',
    'part': PART, 'num_parts': NUM_PARTS, 'tweet_ids': tweet_ids,
    'torch': torch.__version__, 'torchvision': torchvision.__version__, 'timm': timm.__version__,
    'numpy': np.__version__, 'pillow': PIL.__version__,
    'python': platform.python_version(), 'seed': SEED,
    'fine_tuning': 'MMSD image-only sarcasm classifier; train split; validation F1 selection',
}
print(json.dumps({k: v for k, v in manifest.items() if k != 'tweet_ids'}, indent=2))


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_dir = OUTPUT_DIR / 'manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)
manifest_path = manifest_dir / f'part_{PART}.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Cache mismatch: use a new output directory.'
else:
    assert not any((OUTPUT_DIR / f'{i}.npy').exists() for i in tweet_ids), 'This part already has untracked features; use a new output directory.'
    manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')
pending = []
for tweet_id in tweet_ids:
    path = OUTPUT_DIR / f'{tweet_id}.npy'
    if path.exists():
        cached = np.load(path, allow_pickle=False, mmap_mode='r')
        assert cached.shape == (196, 2048) and cached.dtype == np.float32, path
        assert np.isfinite(cached).all(), path
        del cached
    else:
        pending.append(tweet_id)
required = len(pending) * (196 * 2048 * 4 + 128) + 1024**3
assert shutil.disk_usage(OUTPUT_DIR).free >= required, 'Insufficient free disk space (including 1 GiB margin).'


In [ ]:
start = time.perf_counter()
progress = tqdm(total=len(pending), desc=f'Part {PART}', unit='image')
for offset in range(0, len(pending), BATCH_SIZE):
    ids = pending[offset:offset + BATCH_SIZE]
    batch = extract_batch(ids)
    for tweet_id, feature in zip(ids, batch):
        target = OUTPUT_DIR / f'{tweet_id}.npy'
        temporary = target.with_suffix('.npy.tmp')
        with temporary.open('wb') as stream:
            np.save(stream, feature, allow_pickle=False)
        os.replace(temporary, target)
    progress.update(len(ids))
progress.close()
assert all((OUTPUT_DIR / f'{i}.npy').is_file() for i in tweet_ids)
print('Complete:', len(tweet_ids), 'feature files in', OUTPUT_DIR)
print('Elapsed extraction time for this run:', time.perf_counter() - start, 'seconds')


In [ ]:
import shutil

shutil.make_archive("/kaggle/working/output", "zip", "/kaggle/working")